# DeBERTa-v3-base: дообучение для прогноза рейтинга

Проверяем, помогает ли адаптация предобученного трансформера к отзывам. Сохраняем ту же валидационную часть, что в `baselinev2.ipynb` и `mpnet_catboost.ipynb`. Чтобы эксперимент поместился на Apple M1 с 8 ГБ памяти, обучаем классификатор и два верхних слоя энкодера, а остальные веса оставляем замороженными. Используем **все 47 980 обучающих отзывов**, одну эпоху и максимум 128 токенов. На случайном образце 5 000 отзывов этот лимит обрезал около 13,7% текстов. Это ограничение эксперимента, а не свойство датасета в целом.

Числа и таблицы ниже сохранены из предыдущего запуска на настоящих данных. После разбиения длинных ячеек обучение повторно не запускалось.

In [ ]:
import json
import os
import random
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def cache_digest(config: dict) -> str:
    """Короткий устойчивый ключ для JSON-совместимых параметров эксперимента."""
    import json
    from hashlib import sha256

    payload = json.dumps(
        config, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    )
    return sha256(payload.encode("utf-8")).hexdigest()[:12]


seed = 2026

random.seed(seed)

np.random.seed(seed)

torch.manual_seed(seed)

In [ ]:
torch.set_num_threads(4)

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
data_path = root / "data" / "raw" / "train.csv"
if not data_path.exists():
    raise FileNotFoundError("Поместите train.csv в data/raw/")

train = pd.read_csv(data_path)
assert list(train.columns) == ["Id", "Review", "Rating"]

clean = train.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)
groups = clean["Review"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
train_idx, valid_idx = next(splitter.split(clean["Review"], clean["Rating"], groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[valid_idx]))

In [ ]:
y_train = clean["Rating"].iloc[train_idx].to_numpy()
y_valid = clean["Rating"].iloc[valid_idx].to_numpy()
print(f"Обучение: {len(train_idx):,}; валидация: {len(valid_idx):,}")

Обучение: 47,980; валидация: 11,996

## Токенизация

Используем исходный SentencePiece-токенизатор DeBERTa. Метки 1–5 преобразуем в классы 0–4 только на время обучения. Проверочные тексты токенизируются теми же правилами, но их оценки не участвуют в обучении.

In [ ]:
model_id = "microsoft/deberta-v3-base"
model_revision = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
max_length = 128
batch_size = 8
num_epochs = 1
trainable_encoder_layers = 2
encoder_lr = 3e-5
head_lr = 1e-4
warmup_fraction = 0.05
max_grad_norm = 1.0
device = "mps" if torch.backends.mps.is_available() else "cpu"

data_sha256 = sha256(data_path.read_bytes()).hexdigest()
clean_hash = sha256()

In [ ]:
for review, rating in zip(clean["Review"], clean["Rating"]):
    clean_hash.update(review.encode("utf-8"))
    clean_hash.update(b"\0")
    clean_hash.update(bytes((int(rating),)))

split_sha256 = sha256(
    train_idx.astype("<i8").tobytes() + valid_idx.astype("<i8").tobytes()
).hexdigest()

In [ ]:
training_config = {
    "data_sha256": data_sha256,
    "clean_reviews_sha256": clean_hash.hexdigest(),
    "split_sha256": split_sha256,
    "model_id": model_id,
    "model_revision": model_revision,
    "max_length": max_length,
    "use_fast_tokenizer": False,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "trainable_encoder_layers": trainable_encoder_layers,
    "encoder_lr": encoder_lr,
    "head_lr": head_lr,
    "warmup_fraction": warmup_fraction,
    "max_grad_norm": max_grad_norm,
    "optimizer": "AdamW",
    "loss": "cross_entropy",
    "seed": seed,
    "device": device,
    "torch_version": torch.__version__,
    "transformers_version": version("transformers"),
}

training_digest = cache_digest(training_config)

cache_dir = root / "data" / "cache"

In [ ]:
cache_dir.mkdir(parents=True, exist_ok=True)
model_dir = (
    cache_dir / f"deberta_v3_base_top{trainable_encoder_layers}_{training_digest}"
)
model_manifest = model_dir / "training_config.json"

tokenizer = AutoTokenizer.from_pretrained(
    model_id, revision=model_revision, use_fast=False
)
started = perf_counter()
train_tokens = tokenizer(
    clean["Review"].iloc[train_idx].tolist(),
    padding="max_length",
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

In [ ]:
valid_tokens = tokenizer(
    clean["Review"].iloc[valid_idx].tolist(),
    padding="max_length",
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

train_dataset = TensorDataset(
    train_tokens["input_ids"],
    train_tokens["attention_mask"],
    torch.tensor(y_train - 1, dtype=torch.long),
)
valid_dataset = TensorDataset(
    valid_tokens["input_ids"],
    valid_tokens["attention_mask"],
    torch.tensor(y_valid - 1, dtype=torch.long),
)

print(f"Токенизация: {(perf_counter() - started) / 60:.1f} мин; устройство: {device}")

Токенизация: 0.2 мин; устройство: mps

## Дообучение

Фиксируем параметры до оценки: cross-entropy для пяти классов, AdamW, линейное снижение скорости обучения, одна эпоха. Обучаем только два верхних слоя энкодера, классификационную голову и pooler. Кэш модели в `data/cache/` зависит от данных, разбиения, версии модели и зафиксированных параметров обучения; при их изменении обучение запускается заново.

In [ ]:
if model_dir.exists() != model_manifest.exists():
    raise RuntimeError(
        "Кэш DeBERTa неполный: удалите соответствующий каталог из data/cache/"
    )

needs_training = not (model_dir.exists())
if not needs_training:
    if json.loads(model_manifest.read_text(encoding="utf-8")) != training_config:
        raise RuntimeError("Параметры кэша DeBERTa не совпадают с текущим запуском")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir, local_files_only=True
    )
    print("Использована локальная обученная модель")

In [ ]:
if needs_training:
    temporary_dir = model_dir.with_name(model_dir.name + ".partial")
    if temporary_dir.exists():
        raise RuntimeError(f"Остался незавершённый каталог обучения: {temporary_dir}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, revision=model_revision, num_labels=5
    )

    for parameter in model.deberta.parameters():
        parameter.requires_grad = False
    for layer in model.deberta.encoder.layer[-trainable_encoder_layers:]:
        for parameter in layer.parameters():
            parameter.requires_grad = True

    encoder_parameters = [
        parameter
        for layer in model.deberta.encoder.layer[-trainable_encoder_layers:]
        for parameter in layer.parameters()
    ]
    head_parameters = list(model.pooler.parameters()) + list(
        model.classifier.parameters()
    )
    trainable_parameters = encoder_parameters + head_parameters

    print("Обучаемых параметров:", f"{sum(p.numel() for p in trainable_parameters):,}")

In [ ]:
if needs_training:
    model.to(device).train()
    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_parameters, "lr": encoder_lr},
            {"params": head_parameters, "lr": head_lr},
        ],
        foreach=False,
    )
    total_steps = len(loader) * num_epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=round(total_steps * warmup_fraction),
        num_training_steps=total_steps,
    )

    started = perf_counter()
    loss_sum = 0.0

In [ ]:
if needs_training:
    for epoch in range(num_epochs):
        for step, (input_ids, attention_mask, labels) in enumerate(loader, start=1):
            optimizer.zero_grad(set_to_none=True)
            output = model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
                labels=labels.to(device),
            )

            loss = output.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=max_grad_norm)
            optimizer.step()
            scheduler.step()
            loss_sum += loss.item()

            if step % 500 == 0 or step == len(loader):
                print(
                    f"Эпоха {epoch + 1}, шаг {step:,}/{len(loader):,}, средняя loss {loss_sum / step:.4f}, время {(perf_counter() - started) / 60:.1f} мин",
                    flush=True,
                )

    model.eval()
    temporary_dir.mkdir(parents=True)
    model.save_pretrained(temporary_dir, safe_serialization=True)
    tokenizer.save_pretrained(temporary_dir)
    (temporary_dir / "training_config.json").write_text(
        json.dumps(training_config, sort_keys=True, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

In [ ]:
if needs_training:
    os.replace(temporary_dir, model_dir)
    print(f"Обучение: {(perf_counter() - started) / 60:.1f} мин")

_ = model.to(device).eval()

Обучаемых параметров: 14,770,181

Обучение: 67.9 мин

## Оценка

Из вероятностей пяти классов выбираем **медиану распределения**: первую оценку, для которой накопленная вероятность достигла 0,5. Это согласовано с метрикой MAE. Кэш вероятностей привязан к сохранённому файлу весов: при его замене проверочные отзывы прогоняются заново.

In [ ]:
weights_path = model_dir / "model.safetensors"
weights_hash = sha256()
with weights_path.open("rb") as weights_file:
    for block in iter(lambda: weights_file.read(8 * 1024 * 1024), b""):
        weights_hash.update(block)

weights_digest = weights_hash.hexdigest()[:12]
probability_cache = model_dir / f"valid_probabilities_{weights_digest}.npy"

In [ ]:
if probability_cache.exists():
    probabilities = np.load(probability_cache, allow_pickle=False)
    print("Использован локальный кэш вероятностей валидации")
else:
    loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=0)
    probability_batches = []
    started = perf_counter()

    with torch.inference_mode():
        for step, (input_ids, attention_mask, _) in enumerate(loader, start=1):
            logits = model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
            ).logits
            probability_batches.append(torch.softmax(logits, dim=1).cpu().numpy())
            if step % 100 == 0 or step == len(loader):
                print(
                    f"Проверено {min(step * 32, len(valid_dataset)):,} / {len(valid_dataset):,}",
                    flush=True,
                )

    probabilities = np.concatenate(probability_batches)
    np.save(probability_cache, probabilities)
    print(f"Время проверки: {(perf_counter() - started) / 60:.1f} мин")

assert probabilities.shape == (len(y_valid), 5)

In [ ]:
assert np.isfinite(probabilities).all()

predictions = np.argmax(np.cumsum(probabilities, axis=1) >= 0.5, axis=1) + 1
mae = mean_absolute_error(y_valid, predictions)
print(f"MAE DeBERTa-v3-base (два верхних слоя): {mae:.5f}")

by_rating = pd.DataFrame(
    {
        "строк": [int((y_valid == rating).sum()) for rating in range(1, 6)],
        "MAE": [
            mean_absolute_error(
                y_valid[y_valid == rating], predictions[y_valid == rating]
            )
            for rating in range(1, 6)
        ],
    },
    index=pd.Index(range(1, 6), name="Rating"),
)
display(by_rating.round(4))

Время проверки: 13.7 мин

MAE DeBERTa-v3-base (два верхних слоя): 0.14722

| Rating | строк | MAE |
| --- | --- | --- |
| 1 | 3732 | 0.0654 |
| 2 | 326 | 0.9233 |
| 3 | 336 | 1.1220 |
| 4 | 670 | 0.7687 |
| 5 | 6932 | 0.0475 |

## Выводы

- Частично дообученная DeBERTa-v3-base дала **MAE 0,14722** на отложенной части. TF-IDF по словам и символам + 13 числовых признаков + Logistic Regression с `C=4`, `min_df=5` из `baselinev2.ipynb` получил **0,18881 на тех же 11 996 отзывах**; улучшение составляет 0,04159 MAE (около 22,0%). Среднюю MAE пяти фолдов из `baselinev2.ipynb` здесь не используем, поскольку это другая схема оценки. MPNet + CatBoost на этом разбиении получил 0,44325.
- По оценкам 1 и 5 MAE составила 0,0654 и 0,0475. Для редких оценок 2–4 ошибки всё ещё заметно выше: 0,9233 / 1,1220 / 0,7687. Общая метрика не означает, что промежуточные оценки решены хорошо.
- В новом запуске обучение двух верхних слоёв и классификатора на всех 47 980 обучающих отзывах заняло 67,9 мин на Apple M1; проверка 11 996 отзывов — 13,7 мин. Вход ограничен 128 токенами, поэтому часть длинных отзывов обрезается. Полное дообучение всех слоёв и другие настройки в этом эксперименте не проверялись.
- Одиночный CPU-прогноз проверен отдельно на Apple M1 с двумя потоками: медиана 58,35 мс, p95 87,33 мс (30 вызовов после прогрева, тексты длиной 52 / 164 / 717 символов, предобработка включена). DeBERTa укладывается в требование по задержке. Для стандартного Docker-сервиса выбран TF-IDF с артефактом 4,48 МБ; локальный каталог DeBERTa занимает около 740 МБ. DeBERTa сохраняется как более качественный, но более тяжёлый в доставке эксперимент.